# Perron Tree Construction Visualisation for Cubes

**Setup:** $Q = [0,1]^{d-1}$, origin $o = 0$, vertex $v = (\tfrac{1}{2},\ldots,\tfrac{1}{2},1)$.

At the $n$-th dyadic decomposition level, each cell $Q_i = x_i + 2^{-n}[0,1]^{d-1}$ is determined by its minimal corner
$$x_i = \sum_{j=1}^n \beta_j 2^{-j}, \qquad \beta_j \in \{0,1\}^{d-1}.$$

We apply the translation
$$w_i = -\frac{1}{n+1}\sum_{j=1}^n \beta_j 2^{-j}(n-j+2)$$
to each pyramid with base $Q_i$ and apex $v$.

In [ ]:
# @title
# ── Install / import ──────────────────────────────────────────────────────────
# plotly is pre-installed in Colab; numpy too.
import numpy as np

import plotly.graph_objects as go
from itertools import product as iproduct

In [ ]:
# @title
# ══════════════════════════════════════════════════════════════════════════════
# CORE CONSTRUCTION
# ══════════════════════════════════════════════════════════════════════════════

# ── Parameters (edit here) ────────────────────────────────────────────────────
n = 1   # number of dyadic iterations  (try 1, 2, 3, 4)
d = 3   # ambient dimension            (2 or 3)

def compute_cells(n, d):
    """
    Returns a list of dicts, one per dyadic cell:
        'beta'   : array of shape (n, d-1) with entries in {0,1}
        'x_i'    : minimal corner of Q_i  in R^{d-1}
        'w_i'    : translation vector      in R^{d-1}
        'side'   : 2^{-n}  (side length of Q_i)
    """
    dim = d - 1          # dimension of the base cube
    side = 2.0 ** (-n)

    # All multi-indices  beta_1, ..., beta_n  each in {0,1}^{dim}
    # We enumerate them as tuples of length n*dim
    single_step = list(iproduct([0, 1], repeat=dim))  # 2^dim possibilities per step
    all_seqs    = list(iproduct(single_step, repeat=n))  # (2^dim)^n total cells

    cells = []
    for seq in all_seqs:
        # seq  is a tuple of n elements, each in {0,1}^dim
        beta = np.array(seq, dtype=float)   # shape (n, dim)

        # x_i = sum_{j=1}^{n}  beta_j * 2^{-j}   (vector in R^{dim})
        j_vals = np.arange(1, n + 1)              # j = 1 … n
        weights_x = 2.0 ** (-j_vals)              # shape (n,)
        x_i = (beta * weights_x[:, None]).sum(axis=0)  # shape (dim,)

        # w_i = -1/(n+1) * sum_{j=1}^{n}  beta_j * 2^{-j} * (n-j+2)
        weights_w = 2.0 ** (-j_vals) * (n - j_vals + 2)  # shape (n,)
        w_i = -(1.0 / (n + 1)) * (beta * weights_w[:, None]).sum(axis=0)  # shape (dim,)

        cells.append({'beta': beta, 'x_i': x_i, 'w_i': w_i, 'side': side})

    return cells


def pyramid_vertices(x_i, side, v, w_i, d):
    """
    Returns the vertices of the translated pyramid:
      - base corners: x_i + 2^{-n} * {0,1}^{d-1}  (lifted to height 0)
      - apex:         v  (at height 1)
    All shifted by the full translation W_i = (w_i, 0)  [base stays in z=0 plane,
    apex translation acts only on the horizontal components].

    Returns array of shape (2^{d-1}+1, d).
    """
    dim = d - 1
    base_offsets = np.array(list(iproduct([0, 1], repeat=dim)), dtype=float) * side  # (2^dim, dim)

    # Base corners in R^d  (horizontal coords + height 0)
    base_pts = np.hstack([x_i + base_offsets, np.zeros((len(base_offsets), 1))])  # (2^dim, d)

    # Apex in R^d
    apex = np.array(v, dtype=float).reshape(1, d)   # (1, d)

    # Full translation vector in R^d: only horizontal components shift
    W = np.zeros(d)
    W[:dim] = w_i

    verts = np.vstack([base_pts, apex]) + W   # shift everything
    return verts

cells = compute_cells(n, d)

In [ ]:
# @title
# ══════════════════════════════════════════════════════════════════════════════
# PLOTTING  ── d = 2  (2-D:  segments become triangles in the plane)
# ══════════════════════════════════════════════════════════════════════════════
import numpy as np
import plotly.colors as pc
import plotly.graph_objects as go

def plot_2d(n, cells):
    """d=2: base is an interval [x_i, x_i+2^{-n}] at y=0, apex v=(1/2,1)."""
    v = np.array([0.5, 1.0])
    palette = pc.qualitative.Plotly

    fig = go.Figure()

    for k, cell in enumerate(cells):
        verts = pyramid_vertices(cell['x_i'], cell['side'], v, cell['w_i'], d=2)
        # verts: rows = [left-base, right-base, apex], shape (3,2)
        # Close the triangle
        xs = list(verts[:, 0]) + [verts[0, 0]]
        ys = list(verts[:, 1]) + [verts[0, 1]]

        color = palette[k % len(palette)]
        label = f"cell {k}: β={cell['beta'].astype(int).flatten().tolist()}"

        fig.add_trace(go.Scatter(
            x=xs, y=ys,
            mode='lines',
            fill='toself',
            fillcolor=color,
            opacity=0.35,
            line=dict(color=color, width=1.5),
            name=label,
            hovertemplate=(
                f"<b>{label}</b><br>"
                f"x_i = {cell['x_i'].round(4)}<br>"
                f"w_i = {cell['w_i'].round(4)}<extra></extra>"
            )
        ))

    # Draw the original (un-translated) reference triangle lightly
    ref_xs = [0, 1, 0.5, 0]
    ref_ys = [0, 0, 1,   0]
    fig.add_trace(go.Scatter(
        x=ref_xs, y=ref_ys, mode='lines',
        line=dict(color='black', width=2, dash='dot'),
        name='Reference pyramid (Q, v)'
    ))

    fig.update_layout(
        showlegend=False,
        width=800,
        height=700,
        paper_bgcolor='white',
        plot_bgcolor='white',
        xaxis=dict(
            visible=False,
            scaleanchor='y',
            scaleratio=1
        ),
        yaxis=dict(
            visible=False,
            range=[-0.15, 1.2]
        ),
        margin=dict(l=10, r=10, t=10, b=10) # Minimal padding to keep it tight
    )
    fig.show()

In [ ]:
# @title
# ══════════════════════════════════════════════════════════════════════════════
# PLOTTING  ── d = 3  (3-D: base is a square [0,1]^2, apex v=(1/2,1/2,1))
# ══════════════════════════════════════════════════════════════════════════════

import numpy as np
import plotly.colors as pc
import plotly.graph_objects as go

def square_pyramid_mesh(verts):
    """
    verts : array (5, 3)
        verts[0..3] = base corners in order: (0,0),(1,0),(1,1),(0,1)  (scaled)
        verts[4]    = apex

    Returns (x, y, z, i, j, k) for go.Mesh3d  — 6 triangular faces.
    """
    # Reorder base corners so they go around the square
    b = verts[[0, 2, 3, 1]]   # corners in cyclic order
    a = verts[4]               # apex
    pts = np.vstack([b, a])    # 0..3 = base, 4 = apex

    # 4 side faces + 2 triangles for the base quad
    i_idx = [0, 1, 2, 3, 0, 1]
    j_idx = [1, 2, 3, 0, 2, 3]
    k_idx = [4, 4, 4, 4, 3, 0]   # last two close the base

    return pts[:, 0], pts[:, 1], pts[:, 2], i_idx, j_idx, k_idx


def plot_3d(n, cells):
    """d=3: base is a square Q_i ⊂ R^2 × {0}, apex v=(1/2,1/2,1)."""
    v = np.array([0.5, 0.5, 1.0])

    # Use a continuous colorscale to distinguish cells
    N = len(cells)
    colors_rgb = pc.sample_colorscale('Turbo', [k / max(N - 1, 1) for k in range(N)])

    fig = go.Figure()

    for k, cell in enumerate(cells):
        verts = pyramid_vertices(cell['x_i'], cell['side'], v, cell['w_i'], d=3)
        # verts shape: (5, 3)  — 4 base corners + 1 apex
        x, y, z, ii, jj, kk = square_pyramid_mesh(verts)
        color = colors_rgb[k]
        label = f"cell {k}"

        fig.add_trace(go.Mesh3d(
            x=x, y=y, z=z,
            i=ii, j=jj, k=kk,
            color=color,
            opacity=0.40,
            flatshading=True,
            name=label,
            hovertemplate=(
                f"<b>{label}</b><br>"
                f"β = {cell['beta'].astype(int).tolist()}<br>"
                f"x_i = {cell['x_i'].round(4).tolist()}<br>"
                f"w_i = {cell['w_i'].round(4).tolist()}<extra></extra>"
            ),
            showscale=False
        ))

        # Wireframe edges for this pyramid
        b_ord = verts[[0, 2, 3, 1, 0], :]   # base loop (cyclic)
        edge_x, edge_y, edge_z = [], [], []
        # base loop
        for p in b_ord:
            edge_x.append(p[0]); edge_y.append(p[1]); edge_z.append(p[2])
        # lateral edges (base corners → apex)
        apex = verts[4]
        for ci in [0, 2, 3, 1]:
            c = verts[ci]
            edge_x += [c[0], apex[0], None]
            edge_y += [c[1], apex[1], None]
            edge_z += [c[2], apex[2], None]

        fig.add_trace(go.Scatter3d(
            x=edge_x, y=edge_y, z=edge_z,
            mode='lines',
            line=dict(color='black', width=1),
            showlegend=False,
            hoverinfo='skip'
        ))

    # Reference pyramid (un-translated, grey)
    ref_base = np.array([[0,0,0],[1,0,0],[1,1,0],[0,1,0]], dtype=float)
    ref_apex = np.array([[0.5, 0.5, 1.0]])
    ref_pts  = np.vstack([ref_base, ref_apex])
    fig.add_trace(go.Mesh3d(
        x=ref_pts[:,0], y=ref_pts[:,1], z=ref_pts[:,2],
        i=[0,1,2,3,0,1], j=[1,2,3,0,2,3], k=[4,4,4,4,3,0],
        color='lightgrey', opacity=0.12,
        flatshading=True, name='Reference pyramid',
        showscale=False
    ))

    fig.update_layout(
        showlegend=False,
        width=900,
        height=750,
        paper_bgcolor='white',
        plot_bgcolor='white',
        scene=dict(
            xaxis=dict(visible=False, range=[-0.8, 1.3]),
            yaxis=dict(visible=False, range=[-0.8, 1.3]),
            zaxis=dict(visible=False, range=[-0.1, 1.2]),
            bgcolor='white',
            aspectmode='manual',
            aspectratio=dict(x=1, y=1, z=0.8)
        ),
        margin=dict(l=10, r=10, t=10, b=10)
    )
    fig.show()

# In the following cell you can choose how many iterations you want for the 2D and 3D construction. Just edit the $n$ value in the code below:

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# COMBINED RUNNER  ── plots both d=2 and d=3
# ══════════════════════════════════════════════════════════════════════════════

print("=== d = 2 ===")
cells_2d = compute_cells(n=5, d=2)
plot_2d(n, cells_2d)

print("\n=== d = 3 ===")
cells_3d = compute_cells(n=3, d=3)
plot_3d(n, cells_3d)

**Setup:** $Q = [-1/2,1/2]^{d-1}$, origin $o = 0$, vertex $v = (0,\dots,0,1)$.

#The following plots show the 3D Perron tree construction with different iterations.

For each construction we have also added the reflected cones for each pyramid in order to illustrate their disjointness.

In [ ]:
# @title
import numpy as np
import plotly.graph_objects as go

# ─── Global geometry ──────────────────────────────────────────────────────────
SIGMA_MIN = np.array([0.0, 0.0])
SIGMA_MAX = np.array([1.0, 1.0])
V  = np.array([0.5, 0.5, 1.0])   # apex v
O  = np.array([0.5, 0.5, 0.0])   # centre o
O3 = np.array([0.5, 0.5, 0.0])   # o embedded in R³

def get_dyadic_cells(box_min, box_max):
    mid = 0.5 * (box_min + box_max)
    corners_2d = [
        np.array([box_min[0], box_min[1]]),
        np.array([box_max[0], box_min[1]]),
        np.array([box_min[0], box_max[1]]),
        np.array([box_max[0], box_max[1]]),
    ]
    cell_ranges = [
        (np.array([box_min[0], box_min[1]]), np.array([mid[0],     mid[1]])),
        (np.array([mid[0],     box_min[1]]), np.array([box_max[0], mid[1]])),
        (np.array([box_min[0], mid[1]]),     np.array([mid[0],     box_max[1]])),
        (np.array([mid[0],     mid[1]]),     np.array([box_max[0], box_max[1]])),
    ]
    xi_3d = [np.array([c[0], c[1], 0.0]) for c in corners_2d]
    return [(cmin, cmax, xi) for (cmin, cmax), xi in zip(cell_ranges, xi_3d)]

def build_pyramids(n_steps):
    tv = [1.0 / (n_steps - i + 2) for i in range(1, n_steps + 1)]
    Pv = [1.0] + [float(np.prod([1 - tv[j] for j in range(m)]))
                  for m in range(1, n_steps + 1)]

    def recurse(depth, box_min, box_max, acc_w, outer_idx):
        t     = tv[depth - 1]
        cells = get_dyadic_cells(box_min, box_max)
        out   = []
        for idx, (cmin, cmax, xi) in enumerate(cells):
            w_local  = t * (O - xi)
            factor   = Pv[depth - 1]
            new_acc  = acc_w + factor * w_local

            new_outer = idx if depth == n_steps else outer_idx
            if depth == 1:
                bmin3 = np.array([cmin[0] + new_acc[0], cmin[1] + new_acc[1], 0.0])
                bmax3 = np.array([cmax[0] + new_acc[0], cmax[1] + new_acc[1], 0.0])
                out.append((bmin3, bmax3, V + new_acc, new_outer))
            else:
                out.extend(recurse(depth - 1, cmin, cmax, new_acc, new_outer))
        return out

    return recurse(n_steps, SIGMA_MIN, SIGMA_MAX, np.zeros(3), 0), Pv

def compute_solid_geometry(groups_dict, include_reflected=True):
    results = {}
    for g_idx in sorted(groups_dict):
        pyr_list = groups_dict[g_idx]

        # Original component batching collections
        ox, oy, oz = [], [], []
        om_x, om_y, om_z = [], [], []
        om_i, om_j, om_k = [], [], []
        o_offset = 0

        # Reflected component batching collections
        rx, ry, rz = [], [], []
        rm_x, rm_y, rm_z = [], [], []
        rm_i, rm_j, rm_k = [], [], []
        r_offset = 0

        for (bmin, bmax, apex) in pyr_list:
            b = [
                [bmin[0], bmin[1], bmin[2]],
                [bmax[0], bmin[1], bmin[2]],
                [bmax[0], bmax[1], bmin[2]],
                [bmin[0], bmax[1], bmin[2]],
            ]
            a = [apex[0], apex[1], apex[2]]
            pts = b + [a]

            # 1. Wireframe parsing
            edges = [(0,1),(1,2),(2,3),(3,0),(0,4),(1,4),(2,4),(3,4)]
            for (ea, eb) in edges:
                ox += [pts[ea][0], pts[eb][0], None]
                oy += [pts[ea][1], pts[eb][1], None]
                oz += [pts[ea][2], pts[eb][2], None]

            # 2. Solid mesh triangulation parsing (Base + 4 lateral walls)
            for p in pts:
                om_x.append(p[0])
                om_y.append(p[1])
                om_z.append(p[2])
            om_i.extend([o_offset+0, o_offset+0, o_offset+0, o_offset+1, o_offset+2, o_offset+3])
            om_j.extend([o_offset+1, o_offset+2, o_offset+1, o_offset+2, o_offset+3, o_offset+0])
            om_k.extend([o_offset+2, o_offset+3, o_offset+4, o_offset+4, o_offset+4, o_offset+4])
            o_offset += 5

            if include_reflected:
                rb = [[2*apex[i] - b[j][i] for i in range(3)] for j in range(4)]
                rpts = rb + [a]

                for (ea, eb) in edges:
                    rx += [rpts[ea][0], rpts[eb][0], None]
                    ry += [rpts[ea][1], rpts[eb][1], None]
                    rz += [rpts[ea][2], rpts[eb][2], None]

                for p in rpts:
                    rm_x.append(p[0])
                    rm_y.append(p[1])
                    rm_z.append(p[2])
                rm_i.extend([r_offset+0, r_offset+0, r_offset+0, r_offset+1, r_offset+2, r_offset+3])
                rm_j.extend([r_offset+1, r_offset+2, r_offset+1, r_offset+2, r_offset+3, r_offset+0])
                rm_k.extend([r_offset+2, r_offset+3, r_offset+4, r_offset+4, r_offset+4, r_offset+4])
                r_offset += 5

        results[g_idx] = {
            'orig_lines': (ox, oy, oz),
            'orig_mesh': (om_x, om_y, om_z, om_i, om_j, om_k),
            'refl_lines': (rx, ry, rz),
            'refl_mesh': (rm_x, rm_y, rm_z, rm_i, rm_j, rm_k)
        }
    return results

def core_wireframe(Pv, n_steps):
    Pn = Pv[n_steps]
    sigma_corners = [
        np.array([0.0, 0.0, 0.0]), np.array([1.0, 0.0, 0.0]),
        np.array([1.0, 1.0, 0.0]), np.array([0.0, 1.0, 0.0]),
    ]
    base = [Pn * (c - O3) + O3 for c in sigma_corners]
    apex = Pn * (V - O3) + O3
    pts  = base + [apex]
    edges = [(0,1),(1,2),(2,3),(3,0),(0,4),(1,4),(2,4),(3,4)]
    xs, ys, zs = [], [], []
    for (a, b) in edges:
        xs += [pts[a][0], pts[b][0], None]
        ys += [pts[a][1], pts[b][1], None]
        zs += [pts[a][2], pts[b][2], None]
    return xs, ys, zs

def original_T_wireframe():
    sigma_corners = [
        np.array([0.0, 0.0, 0.0]), np.array([1.0, 0.0, 0.0]),
        np.array([1.0, 1.0, 0.0]), np.array([0.0, 1.0, 0.0]),
    ]
    pts  = sigma_corners + [V]
    edges = [(0,1),(1,2),(2,3),(3,0),(0,4),(1,4),(2,4),(3,4)]
    xs, ys, zs = [], [], []
    for (a, b) in edges:
        xs += [pts[a][0], pts[b][0], None]
        ys += [pts[a][1], pts[b][1], None]
        zs += [pts[a][2], pts[b][2], None]
    return xs, ys, zs

# ─── Color Palettes (Lighter Shading vs Darker Outlines) ──────────────────────
GROUP_COLORS_LIGHT = ['#f28482', '#f7b785', '#84cbc0', '#8fa3f7'] # Lighter interior walls
GROUP_COLORS_DARK  = ['#b71c1c', '#e65100', '#004d40', '#0d47a1'] # Darker edge wireframes

MINIMAL_LAYOUT = dict(
    paper_bgcolor='white', plot_bgcolor='white', margin=dict(l=0, r=0, b=0, t=0),
    scene=dict(
        xaxis=dict(visible=False), yaxis=dict(visible=False), zaxis=dict(visible=False),
        camera=dict(eye=dict(x=1.9, y=-1.6, z=1.3)), aspectmode='data'
    )
)

# ─── Plotting execution loop n = 1 to 5 ───────────────────────────────────────
for n_steps in range(1, 6):
    pyramids, Pv = build_pyramids(n_steps)
    groups = {0: [], 1: [], 2: [], 3: []}
    for (bmin, bmax, apex, g) in pyramids:
        groups[g].append((bmin, bmax, apex))

    geom_data = compute_solid_geometry(groups, include_reflected=True)
    cxs, cys, czs = core_wireframe(Pv, n_steps)
    T_xs, T_ys, T_zs = original_T_wireframe()

    # ──────────────────────────────────────────────────────────────────────────
    # VARIANT A: WITHOUT REFLECTED PYRAMIDS
    # ──────────────────────────────────────────────────────────────────────────
    print(f"Generating Plots for n={n_steps}...")
    fig_no_refl = go.Figure()

    # Outer Structure Reference (Original T Frame) - Updated to Solid Black Wireframe
    fig_no_refl.add_trace(go.Scatter3d(
        x=T_xs, y=T_ys, z=T_zs, mode='lines',
        line=dict(color='black', width=3), opacity=1.0, showlegend=False
    ))

    for g_idx in geom_data:
        # Original Solid Interior Walls
        mx, my, mz, mi, mj, mk = geom_data[g_idx]['orig_mesh']
        fig_no_refl.add_trace(go.Mesh3d(
            x=mx, y=my, z=mz, i=mi, j=mj, k=mk,
            color=GROUP_COLORS_LIGHT[g_idx], opacity=0.4, showlegend=False,
            lighting=dict(ambient=0.6, diffuse=0.5, roughness=0.5)
        ))
        # Original Dark Wireframe
        lx, ly, lz = geom_data[g_idx]['orig_lines']
        fig_no_refl.add_trace(go.Scatter3d(
            x=lx, y=ly, z=lz, mode='lines',
            line=dict(color=GROUP_COLORS_DARK[g_idx], width=2.5 if n_steps < 3 else 1.2), opacity=0.9, showlegend=False
        ))

    # Core Wireframe C(P_n) - Updated to contrasting Bright Red
    fig_no_refl.add_trace(go.Scatter3d(
        x=cxs, y=cys, z=czs, mode='lines',
        line=dict(color='#FF0000', width=6), opacity=1.0, showlegend=False
    ))

    fig_no_refl.update_layout(MINIMAL_LAYOUT)
    fig_no_refl.show()

    # ──────────────────────────────────────────────────────────────────────────
    # VARIANT B: WITH REFLECTED PYRAMIDS
    # ──────────────────────────────────────────────────────────────────────────
    fig_with_refl = go.Figure()

    # Outer Structure Reference (Original T Frame) - Updated to Solid Black Wireframe
    fig_with_refl.add_trace(go.Scatter3d(
        x=T_xs, y=T_ys, z=T_zs, mode='lines',
        line=dict(color='black', width=3), opacity=1.0, showlegend=False
    ))

    for g_idx in geom_data:
        # 1. Original Solid + Wireframe
        mx, my, mz, mi, mj, mk = geom_data[g_idx]['orig_mesh']
        fig_with_refl.add_trace(go.Mesh3d(
            x=mx, y=my, z=mz, i=mi, j=mj, k=mk,
            color=GROUP_COLORS_LIGHT[g_idx], opacity=0.4, showlegend=False,
            lighting=dict(ambient=0.6, diffuse=0.5, roughness=0.5)
        ))
        lx, ly, lz = geom_data[g_idx]['orig_lines']
        fig_with_refl.add_trace(go.Scatter3d(
            x=lx, y=ly, z=lz, mode='lines',
            line=dict(color=GROUP_COLORS_DARK[g_idx], width=2.5 if n_steps < 3 else 1.2), opacity=0.9, showlegend=False
        ))

        # 2. Reflected Solid + Wireframe
        rmx, rmy, rmz, rmi, rmj, rmk = geom_data[g_idx]['refl_mesh']
        fig_with_refl.add_trace(go.Mesh3d(
            x=rmx, y=rmy, z=rmz, i=rmi, j=rmj, k=rmk,
            color=GROUP_COLORS_LIGHT[g_idx], opacity=0.25, showlegend=False,
            lighting=dict(ambient=0.6, diffuse=0.5, roughness=0.5)
        ))
        rlx, rly, rlz = geom_data[g_idx]['refl_lines']
        fig_with_refl.add_trace(go.Scatter3d(
            x=rlx, y=rly, z=rlz, mode='lines',
            line=dict(color=GROUP_COLORS_DARK[g_idx], width=1.5 if n_steps < 3 else 0.8, dash='dash'), opacity=0.6, showlegend=False
        ))

    # Core Wireframe C(P_n) - Updated to contrasting Bright Red
    fig_with_refl.add_trace(go.Scatter3d(
        x=cxs, y=cys, z=czs, mode='lines',
        line=dict(color='#FF0000', width=6), opacity=1.0, showlegend=False
    ))

    fig_with_refl.update_layout(MINIMAL_LAYOUT)
    fig_with_refl.show()

# Illustration of how one can insert a $\delta$-tube with $\delta\sim 2^{-n}$ into each pyramid of our $n$-iterations Perron tree construction

For each construction we have also added a second picture showing the reflected pyramids and tubes. The number of iterations can be changed directly in the code below:

In [ ]:
# @title
import numpy as np
import plotly.graph_objects as go

# ─── Global geometry ──────────────────────────────────────────────────────────
SIGMA_MIN = np.array([0.0, 0.0])
SIGMA_MAX = np.array([1.0, 1.0])
# Stretched in the z-direction so the pyramids have height 2
V  = np.array([0.5, 0.5, 2.0])
O  = np.array([0.5, 0.5, 0.0])
O3 = np.array([0.5, 0.5, 0.0])

def get_dyadic_cells(box_min, box_max):
    mid = 0.5 * (box_min + box_max)
    corners_2d = [
        np.array([box_min[0], box_min[1]]),
        np.array([box_max[0], box_min[1]]),
        np.array([box_min[0], box_max[1]]),
        np.array([box_max[0], box_max[1]]),
    ]
    cell_ranges = [
        (np.array([box_min[0], box_min[1]]), np.array([mid[0],     mid[1]])),
        (np.array([mid[0],     box_min[1]]), np.array([box_max[0], mid[1]])),
        (np.array([box_min[0], mid[1]]),     np.array([mid[0],     box_max[1]])),
        (np.array([mid[0],     mid[1]]),     np.array([box_max[0], box_max[1]])),
    ]
    xi_3d = [np.array([c[0], c[1], 0.0]) for c in corners_2d]
    return [(cmin, cmax, xi) for (cmin, cmax), xi in zip(cell_ranges, xi_3d)]

def build_pyramids(n_steps):
    tv = [1.0 / (n_steps - i + 2) for i in range(1, n_steps + 1)]
    Pv = [1.0] + [float(np.prod([1 - tv[j] for j in range(m)]))
                  for m in range(1, n_steps + 1)]

    def recurse(depth, box_min, box_max, acc_w, outer_idx):
        t     = tv[depth - 1]
        cells = get_dyadic_cells(box_min, box_max)
        out   = []
        for idx, (cmin, cmax, xi) in enumerate(cells):
            w_local  = t * (O - xi)
            factor   = Pv[depth - 1]
            new_acc  = acc_w + factor * w_local

            new_outer = idx if depth == n_steps else outer_idx
            if depth == 1:
                bmin3 = np.array([cmin[0] + new_acc[0], cmin[1] + new_acc[1], 0.0])
                bmax3 = np.array([cmax[0] + new_acc[0], cmax[1] + new_acc[1], 0.0])
                out.append((bmin3, bmax3, V + new_acc, new_outer))
            else:
                out.extend(recurse(depth - 1, cmin, cmax, new_acc, new_outer))
        return out

    return recurse(n_steps, SIGMA_MIN, SIGMA_MAX, np.zeros(3), 0), Pv

def compute_wireframes(groups_dict):
    """Calculates strictly the wireframes of the pyramids."""
    results = {}
    for g_idx in sorted(groups_dict):
        pyr_list = groups_dict[g_idx]
        ox, oy, oz = [], [], []
        rx, ry, rz = [], [], []

        for (bmin, bmax, apex) in pyr_list:
            b = [
                [bmin[0], bmin[1], bmin[2]],
                [bmax[0], bmin[1], bmin[2]],
                [bmax[0], bmax[1], bmin[2]],
                [bmin[0], bmax[1], bmin[2]],
            ]
            a = [apex[0], apex[1], apex[2]]
            pts = b + [a]

            edges = [(0,1),(1,2),(2,3),(3,0),(0,4),(1,4),(2,4),(3,4)]
            for (ea, eb) in edges:
                ox += [pts[ea][0], pts[eb][0], None]
                oy += [pts[ea][1], pts[eb][1], None]
                oz += [pts[ea][2], pts[eb][2], None]

            # Reflected coordinates
            rb = [[2*apex[i] - b[j][i] for i in range(3)] for j in range(4)]
            rpts = rb + [a]

            for (ea, eb) in edges:
                rx += [rpts[ea][0], rpts[eb][0], None]
                ry += [rpts[ea][1], rpts[eb][1], None]
                rz += [rpts[ea][2], rpts[eb][2], None]

        results[g_idx] = {
            'orig_lines': (ox, oy, oz),
            'refl_lines': (rx, ry, rz),
        }
    return results

def compute_tubes(groups_dict):
    """Generates the embedded tube meshes via affine mapping."""
    results = {}

    # 1. Define standard tube in the reference space (Radius 1/8, Height 1)
    N_theta = 20
    theta = np.linspace(0, 2*np.pi, N_theta, endpoint=False)

    r = 1.0 / 8.0  # Reduced radius to ensure internal containment

    ref_pts = []
    # Bottom ring (z=0)
    for th in theta:
        ref_pts.append((0.5 + r * np.cos(th), 0.5 + r * np.sin(th), 0.0))
    # Top ring (z=1)
    for th in theta:
        ref_pts.append((0.5 + r * np.cos(th), 0.5 + r * np.sin(th), 1.0))

    # Cap centers
    idx_c_bottom = len(ref_pts)
    ref_pts.append((0.5, 0.5, 0.0))
    idx_c_top = len(ref_pts)
    ref_pts.append((0.5, 0.5, 1.0))

    for g_idx in sorted(groups_dict):
        pyr_list = groups_dict[g_idx]
        om_x, om_y, om_z, om_i, om_j, om_k = [], [], [], [], [], []
        rm_x, rm_y, rm_z, rm_i, rm_j, rm_k = [], [], [], [], [], []

        o_offset = 0
        r_offset = 0

        for (bmin, bmax, apex) in pyr_list:
            dx = bmax[0] - bmin[0]
            dy = bmax[1] - bmin[1]
            mx = bmin[0] + dx / 2.0
            my = bmin[1] + dy / 2.0

            # 2. Map standard tube through the exact IFS transformation
            for (x, y, z) in ref_pts:
                # Affine map shearing logic mapped from Iterated Function System
                xp = bmin[0] + x * dx + z * (apex[0] - mx) / 2.0
                yp = bmin[1] + y * dy + z * (apex[1] - my) / 2.0
                zp = z

                om_x.append(xp)
                om_y.append(yp)
                om_z.append(zp)

                # Reflection mapped through apex
                rm_x.append(2 * apex[0] - xp)
                rm_y.append(2 * apex[1] - yp)
                rm_z.append(2 * apex[2] - zp)

            # 3. Triangulate the meshes
            for i in range(N_theta):
                next_i = (i + 1) % N_theta

                # Side walls + Caps
                for i_arr, off in [(om_i, o_offset), (rm_i, r_offset)]:
                    i_arr.extend([off + i, off + next_i, off + i, off + i + N_theta])
                for j_arr, off in [(om_j, o_offset), (rm_j, r_offset)]:
                    j_arr.extend([off + next_i, off + next_i + N_theta, off + next_i, off + next_i + N_theta])
                for k_arr, off in [(om_k, o_offset), (rm_k, r_offset)]:
                    k_arr.extend([off + i + N_theta, off + i + N_theta, off + idx_c_bottom, off + idx_c_top])

            o_offset += len(ref_pts)
            r_offset += len(ref_pts)

        results[g_idx] = {
            'orig_tube_mesh': (om_x, om_y, om_z, om_i, om_j, om_k),
            'refl_tube_mesh': (rm_x, rm_y, rm_z, rm_i, rm_j, rm_k),
        }
    return results

# ─── Color Palettes ───────────────────────────────────────────────────────────
GROUP_COLORS_LIGHT = ['#f28482', '#f7b785', '#84cbc0', '#8fa3f7']
GROUP_COLORS_DARK  = ['#b71c1c', '#e65100', '#004d40', '#0d47a1']

MINIMAL_LAYOUT = dict(
    paper_bgcolor='white', plot_bgcolor='white', margin=dict(l=0, r=0, b=0, t=0),
    scene=dict(
        xaxis=dict(visible=False), yaxis=dict(visible=False), zaxis=dict(visible=False),
        camera=dict(eye=dict(x=1.9, y=-1.6, z=1.3)), aspectmode='data'
    )
)

#===============================================================================
# ─── Plotting execution for n = 3 ─────────────────────────────────────────────
#===============================================================================

if __name__ == "__main__":
    n_steps = 2

    pyramids, Pv = build_pyramids(n_steps)
    groups = {0: [], 1: [], 2: [], 3: []}
    for (bmin, bmax, apex, g) in pyramids:
        groups[g].append((bmin, bmax, apex))

    geom_data = compute_wireframes(groups)
    tube_data = compute_tubes(groups)

    # ──────────────────────────────────────────────────────────────────────────
    # PLOT 1: WITHOUT REFLECTIONS
    # ──────────────────────────────────────────────────────────────────────────
    print(f"Generating Plot 1 (No Reflections) for n={n_steps}...")
    fig1 = go.Figure()

    for g_idx in geom_data:
        # 1. Original Pyramid Wireframes
        lx, ly, lz = geom_data[g_idx]['orig_lines']
        fig1.add_trace(go.Scatter3d(
            x=lx, y=ly, z=lz, mode='lines',
            line=dict(color=GROUP_COLORS_DARK[g_idx], width=1.5), opacity=0.6, showlegend=False
        ))

        # 2. Original Tube Meshes
        mx, my, mz, mi, mj, mk = tube_data[g_idx]['orig_tube_mesh']
        fig1.add_trace(go.Mesh3d(
            x=mx, y=my, z=mz, i=mi, j=mj, k=mk,
            color=GROUP_COLORS_LIGHT[g_idx], opacity=0.9, showlegend=False,
            lighting=dict(ambient=0.6, diffuse=0.5, roughness=0.5)
        ))

    fig1.update_layout(MINIMAL_LAYOUT)
    fig1.show()

    # ──────────────────────────────────────────────────────────────────────────
    # PLOT 2: WITH REFLECTIONS
    # ──────────────────────────────────────────────────────────────────────────
    print(f"Generating Plot 2 (With Reflections) for n={n_steps}...")
    fig2 = go.Figure()

    for g_idx in geom_data:
        # 1. Original Pyramid Wireframes
        lx, ly, lz = geom_data[g_idx]['orig_lines']
        fig2.add_trace(go.Scatter3d(
            x=lx, y=ly, z=lz, mode='lines',
            line=dict(color=GROUP_COLORS_DARK[g_idx], width=1.5), opacity=0.6, showlegend=False
        ))

        # 2. Reflected Pyramid Wireframes
        rlx, rly, rlz = geom_data[g_idx]['refl_lines']
        fig2.add_trace(go.Scatter3d(
            x=rlx, y=rly, z=rlz, mode='lines',
            line=dict(color=GROUP_COLORS_DARK[g_idx], width=1.0, dash='dash'), opacity=0.4, showlegend=False
        ))

        # 3. Original Tube Meshes
        mx, my, mz, mi, mj, mk = tube_data[g_idx]['orig_tube_mesh']
        fig2.add_trace(go.Mesh3d(
            x=mx, y=my, z=mz, i=mi, j=mj, k=mk,
            color=GROUP_COLORS_LIGHT[g_idx], opacity=0.9, showlegend=False,
            lighting=dict(ambient=0.6, diffuse=0.5, roughness=0.5)
        ))

        # 4. Reflected Tube Meshes
        rmx, rmy, rmz, rmi, rmj, rmk = tube_data[g_idx]['refl_tube_mesh']
        fig2.add_trace(go.Mesh3d(
            x=rmx, y=rmy, z=rmz, i=rmi, j=rmj, k=rmk,
            color=GROUP_COLORS_LIGHT[g_idx], opacity=0.4, showlegend=False,
            lighting=dict(ambient=0.6, diffuse=0.5, roughness=0.5)
        ))

    fig2.update_layout(MINIMAL_LAYOUT)
    fig2.show()

In [ ]:
# @title
import numpy as np
import plotly.graph_objects as go

# ─── Global geometry ──────────────────────────────────────────────────────────
SIGMA_MIN = np.array([0.0, 0.0])
SIGMA_MAX = np.array([1.0, 1.0])
# Stretched in the z-direction so the pyramids have height 2
V  = np.array([0.5, 0.5, 2.0])
O  = np.array([0.5, 0.5, 0.0])

def get_dyadic_cells(box_min, box_max):
    mid = 0.5 * (box_min + box_max)
    corners_2d = [
        np.array([box_min[0], box_min[1]]),
        np.array([box_max[0], box_min[1]]),
        np.array([box_min[0], box_max[1]]),
        np.array([box_max[0], box_max[1]]),
    ]
    cell_ranges = [
        (np.array([box_min[0], box_min[1]]), np.array([mid[0],     mid[1]])),
        (np.array([mid[0],     box_min[1]]), np.array([box_max[0], mid[1]])),
        (np.array([box_min[0], mid[1]]),     np.array([mid[0],     box_max[1]])),
        (np.array([mid[0],     mid[1]]),     np.array([box_max[0], box_max[1]])),
    ]
    xi_3d = [np.array([c[0], c[1], 0.0]) for c in corners_2d]
    return [(cmin, cmax, xi) for (cmin, cmax), xi in zip(cell_ranges, xi_3d)]

# 1. Standard translated pyramids (IFS)
def build_pyramids(n_steps):
    tv = [1.0 / (n_steps - i + 2) for i in range(1, n_steps + 1)]
    Pv = [1.0] + [float(np.prod([1 - tv[j] for j in range(m)]))
                  for m in range(1, n_steps + 1)]

    def recurse(depth, box_min, box_max, acc_w, outer_idx):
        t     = tv[depth - 1]
        cells = get_dyadic_cells(box_min, box_max)
        out   = []
        for idx, (cmin, cmax, xi) in enumerate(cells):
            w_local  = t * (O - xi)
            factor   = Pv[depth - 1]
            new_acc  = acc_w + factor * w_local

            new_outer = idx if depth == n_steps else outer_idx
            if depth == 1:
                bmin3 = np.array([cmin[0] + new_acc[0], cmin[1] + new_acc[1], 0.0])
                bmax3 = np.array([cmax[0] + new_acc[0], cmax[1] + new_acc[1], 0.0])
                out.append((bmin3, bmax3, V + new_acc, new_outer))
            else:
                out.extend(recurse(depth - 1, cmin, cmax, new_acc, new_outer))
        return out

    return recurse(n_steps, SIGMA_MIN, SIGMA_MAX, np.zeros(3), 0)

# 2. Untranslated base subdivision
def build_pyramids_no_translation(n_steps):
    def get_cells(depth, box_min, box_max, outer_idx):
        if depth == 1:
            return [(np.array([box_min[0], box_min[1], 0.0]),
                     np.array([box_max[0], box_max[1], 0.0]),
                     V, outer_idx)]
        cells = get_dyadic_cells(box_min, box_max)
        out = []
        for idx, (cmin, cmax, xi) in enumerate(cells):
            new_outer = idx if depth == n_steps else outer_idx
            out.extend(get_cells(depth - 1, cmin, cmax, new_outer))
        return out
    return get_cells(n_steps, SIGMA_MIN, SIGMA_MAX, 0)

def compute_wireframes(groups_dict):
    """Calculates strictly the wireframes of the bounding pyramids."""
    results = {}
    for g_idx in sorted(groups_dict):
        pyr_list = groups_dict[g_idx]
        ox, oy, oz = [], [], []
        rx, ry, rz = [], [], []

        for (bmin, bmax, apex) in pyr_list:
            b = [
                [bmin[0], bmin[1], bmin[2]],
                [bmax[0], bmin[1], bmin[2]],
                [bmax[0], bmax[1], bmin[2]],
                [bmin[0], bmax[1], bmin[2]],
            ]
            a = [apex[0], apex[1], apex[2]]
            pts = b + [a]

            edges = [(0,1),(1,2),(2,3),(3,0),(0,4),(1,4),(2,4),(3,4)]
            for (ea, eb) in edges:
                ox += [pts[ea][0], pts[eb][0], None]
                oy += [pts[ea][1], pts[eb][1], None]
                oz += [pts[ea][2], pts[eb][2], None]

            # Reflected coordinates
            rb = [[2*apex[i] - b[j][i] for i in range(3)] for j in range(4)]
            rpts = rb + [a]

            for (ea, eb) in edges:
                rx += [rpts[ea][0], rpts[eb][0], None]
                ry += [rpts[ea][1], rpts[eb][1], None]
                rz += [rpts[ea][2], rpts[eb][2], None]

        results[g_idx] = {
            'orig_lines': (ox, oy, oz),
            'refl_lines': (rx, ry, rz),
        }
    return results

def compute_tubes(groups_dict, radius=1/8):
    """Generates the embedded tube meshes via affine mapping."""
    results = {}

    # 1. Define standard tube in the reference space
    N_theta = 20
    theta = np.linspace(0, 2*np.pi, N_theta, endpoint=False)

    ref_pts = []
    for th in theta:
        ref_pts.append((0.5 + radius * np.cos(th), 0.5 + radius * np.sin(th), 0.0))
    for th in theta:
        ref_pts.append((0.5 + radius * np.cos(th), 0.5 + radius * np.sin(th), 1.0))

    idx_c_bottom = len(ref_pts)
    ref_pts.append((0.5, 0.5, 0.0))
    idx_c_top = len(ref_pts)
    ref_pts.append((0.5, 0.5, 1.0))

    for g_idx in sorted(groups_dict):
        pyr_list = groups_dict[g_idx]
        om_x, om_y, om_z, om_i, om_j, om_k = [], [], [], [], [], []
        rm_x, rm_y, rm_z, rm_i, rm_j, rm_k = [], [], [], [], [], []

        o_offset = 0
        r_offset = 0

        for (bmin, bmax, apex) in pyr_list:
            dx = bmax[0] - bmin[0]
            dy = bmax[1] - bmin[1]
            mx = bmin[0] + dx / 2.0
            my = bmin[1] + dy / 2.0

            # 2. Map standard tube through the exact transformation
            for (x, y, z) in ref_pts:
                xp = bmin[0] + x * dx + z * (apex[0] - mx) / 2.0
                yp = bmin[1] + y * dy + z * (apex[1] - my) / 2.0
                zp = z

                om_x.append(xp)
                om_y.append(yp)
                om_z.append(zp)

                # Reflection
                rx, ry, rz = 2 * apex[0] - xp, 2 * apex[1] - yp, 2 * apex[2] - zp
                rm_x.append(rx)
                rm_y.append(ry)
                rm_z.append(rz)

            # 3. Triangulate the meshes
            for i in range(N_theta):
                next_i = (i + 1) % N_theta

                for i_arr, off in [(om_i, o_offset), (rm_i, r_offset)]:
                    i_arr.extend([off + i, off + next_i, off + i, off + i + N_theta])
                for j_arr, off in [(om_j, o_offset), (rm_j, r_offset)]:
                    j_arr.extend([off + next_i, off + next_i + N_theta, off + next_i, off + next_i + N_theta])
                for k_arr, off in [(om_k, o_offset), (rm_k, r_offset)]:
                    k_arr.extend([off + i + N_theta, off + i + N_theta, off + idx_c_bottom, off + idx_c_top])

            o_offset += len(ref_pts)
            r_offset += len(ref_pts)

        results[g_idx] = {
            'orig_tube_mesh': (om_x, om_y, om_z, om_i, om_j, om_k),
            'refl_tube_mesh': (rm_x, rm_y, rm_z, rm_i, rm_j, rm_k)
        }
    return results

# ─── Color Palettes & Layout ──────────────────────────────────────────────────
GROUP_COLORS_LIGHT = ['#f28482', '#f7b785', '#84cbc0', '#8fa3f7']
GROUP_COLORS_DARK  = ['#b71c1c', '#e65100', '#004d40', '#0d47a1']

def get_layout(title_text):
    return dict(
        title=dict(text=title_text, x=0.5, font=dict(size=16)),
        paper_bgcolor='white', plot_bgcolor='white', margin=dict(l=0, r=0, b=0, t=40),
        scene=dict(
            xaxis=dict(visible=False), yaxis=dict(visible=False), zaxis=dict(visible=False),
            camera=dict(eye=dict(x=1.8, y=-1.8, z=1.0)), aspectmode='data'
        )
    )

def plot_scene(geom_data, tube_data, title):
    fig = go.Figure()

    for g_idx in geom_data:
        # Original Pyramid Wireframes
        lx, ly, lz = geom_data[g_idx]['orig_lines']
        fig.add_trace(go.Scatter3d(x=lx, y=ly, z=lz, mode='lines',
            line=dict(color=GROUP_COLORS_DARK[g_idx], width=1.5), opacity=0.3, showlegend=False))

        # Reflected Pyramid Wireframes
        rlx, rly, rlz = geom_data[g_idx]['refl_lines']
        fig.add_trace(go.Scatter3d(x=rlx, y=rly, z=rlz, mode='lines',
            line=dict(color=GROUP_COLORS_DARK[g_idx], width=1.0, dash='dash'), opacity=0.2, showlegend=False))

        # Original Tube Meshes
        mx, my, mz, mi, mj, mk = tube_data[g_idx]['orig_tube_mesh']
        fig.add_trace(go.Mesh3d(x=mx, y=my, z=mz, i=mi, j=mj, k=mk,
            color=GROUP_COLORS_LIGHT[g_idx], opacity=0.9, showlegend=False,
            lighting=dict(ambient=0.6, diffuse=0.5, roughness=0.5)))

        # Reflected Tube Meshes (Styled identical to the original base tubes)
        rmx, rmy, rmz, rmi, rmj, rmk = tube_data[g_idx]['refl_tube_mesh']
        fig.add_trace(go.Mesh3d(x=rmx, y=rmy, z=rmz, i=rmi, j=rmj, k=rmk,
            color=GROUP_COLORS_LIGHT[g_idx], opacity=0.9, showlegend=False,
            lighting=dict(ambient=0.6, diffuse=0.5, roughness=0.5)))

    fig.update_layout(get_layout(title))
    fig.show()

#===============================================================================
# ─── Plotting execution for n = 3 ─────────────────────────────────────────────
#===============================================================================

if __name__ == "__main__":
    n_steps = 2
    tube_radius = 1 / 2**n_steps

    # --- PLOT 1: Standard Translated Pyramids ---
    pyramids_trans = build_pyramids(n_steps)
    groups_trans = {0: [], 1: [], 2: [], 3: []}
    for (bmin, bmax, apex, g) in pyramids_trans:
        groups_trans[g].append((bmin, bmax, apex))

    geom_data_trans = compute_wireframes(groups_trans)
    tube_data_trans = compute_tubes(groups_trans, radius=tube_radius)

    print(f"Generating Plot 1: Translated IFS (n={n_steps})")
    plot_scene(geom_data_trans, tube_data_trans, "n=3 | Translated (Standard IFS) | Matching Reflections")

    # --- PLOT 2: Untranslated Base Subdivision ---
    pyramids_untrans = build_pyramids_no_translation(n_steps)
    groups_untrans = {0: [], 1: [], 2: [], 3: []}
    for (bmin, bmax, apex, g) in pyramids_untrans:
        groups_untrans[g].append((bmin, bmax, apex))

    geom_data_untrans = compute_wireframes(groups_untrans)
    tube_data_untrans = compute_tubes(groups_untrans, radius=tube_radius)

    print(f"Generating Plot 2: Untranslated Base Subdivision (n={n_steps})")
    plot_scene(geom_data_untrans, tube_data_untrans, "n=3 | Untranslated Pyramids | Matching Reflections")

In [ ]:
# @title
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from matplotlib.collections import PatchCollection

# ─── Geometry Helpers ─────────────────────────────────────────────────────────
def get_2d_cells(box_min, box_max):
    mid = 0.5 * (box_min + box_max)
    return [
        (np.array([box_min[0], box_min[1]]), np.array([mid[0], mid[1]])),
        (np.array([mid[0], box_min[1]]), np.array([box_max[0], mid[1]])),
        (np.array([box_min[0], mid[1]]), np.array([mid[0], box_max[1]])),
        (np.array([mid[0], mid[1]]), np.array([box_max[0], box_max[1]])),
    ]

# ─── Recursive Generator for Reflected Bases (2A - B) ─────────────────────────
def generate_refl_bases(depth, n_steps, box_min, box_max, acc_w, tv, Pv):
    if depth == 0:
        bmin = box_min + acc_w
        bmax = box_max + acc_w
        apex_2d = np.array([0.5, 0.5]) + acc_w
        rmin = 2 * apex_2d - bmax
        rmax = 2 * apex_2d - bmin
        yield (rmin, rmax)
        return

    t = tv[n_steps - depth]
    factor = Pv[n_steps - depth]
    cells = get_2d_cells(box_min, box_max)

    for cmin, cmax in cells:
        xi = cmin
        w_local = t * (np.array([0.5, 0.5]) - xi)
        new_acc = acc_w + factor * w_local

        yield from generate_refl_bases(depth - 1, n_steps, cmin, cmax, new_acc, tv, Pv)

# ─── Render the 3x3 Grid ──────────────────────────────────────────────────────
print("Compiling 3x3 grid with uniform blue intensity...")

fig, axes = plt.subplots(2, 2, figsize=(15, 15), dpi=200)
axes = axes.flatten()

# CONSTANT INTENSITY: Every square across all values of n starts identical
UNIFORM_ALPHA = 0.4

for n_steps in range(1, 5):
    tv = [1.0 / (n_steps - i + 2) for i in range(1, n_steps + 1)]
    Pv = [1.0] + [float(np.prod([1 - tv[j] for j in range(m)])) for m in range(1, n_steps + 1)]

    bases = list(generate_refl_bases(n_steps, n_steps, np.array([0.0, 0.0]), np.array([1.0, 1.0]), np.array([0.0, 0.0]), tv, Pv))

    g_min = np.array([float('inf'), float('inf')])
    g_max = np.array([float('-inf'), float('-inf')])

    patches = []
    for rmin, rmax in bases:
        g_min = np.minimum(g_min, rmin)
        g_max = np.maximum(g_max, rmax)

        width = rmax[0] - rmin[0]
        height = rmax[1] - rmin[1]
        patches.append(Rectangle((rmin[0], rmin[1]), width, height))

    ax = axes[n_steps - 1]

    # Applied uniformly to all subplots
    collection = PatchCollection(patches, facecolor='#4361ee', alpha=UNIFORM_ALPHA, linewidth=0)
    ax.add_collection(collection)

    # Draw the original bounding Unit Square [0,1]x[0,1]
    ax.add_patch(Rectangle((0, 0), 1, 1, fill=False, edgecolor='#111111', linestyle='--', linewidth=1.5))

    # Auto-viewport fitting
    margin_x = (g_max[0] - g_min[0]) * 0.1
    margin_y = (g_max[1] - g_min[1]) * 0.1
    ax.set_xlim(g_min[0] - margin_x, g_max[0] + margin_x)
    ax.set_ylim(g_min[1] - margin_y, g_max[1] + margin_y)

    ax.set_aspect('equal')
    ax.axis('off')

    if n_steps == 1:
        ax.set_title(f"n = {n_steps} (4 bases)", fontsize=14, fontweight='bold')
    else:
        ax.set_title(f"n = {n_steps} ({4**n_steps:,} bases)", fontsize=14, fontweight='bold')

plt.suptitle("Bases of Reflected Pyramids", fontsize=20, y=0.98)
plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.show()